# 04 — Recall Deep Dive: TEMPR Multi-Strategy Retrieval

This notebook covers `recall()` in depth:
- The four TEMPR retrieval strategies
- Budget levels (low/mid/high)
- Type filtering (world/experience/observation)
- Tag and metadata filtering
- Chunk retrieval for evidence
- Entity extraction in results
- Debugging with trace mode

In [ ]:
from hindsight_client import Hindsight
from datetime import datetime, timezone

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-recall"

client = Hindsight(base_url=HINDSIGHT_URL)

# Fresh bank with diverse test data
client.create_bank(
    bank_id=BANK_ID,
    name="Recall Tutorial",
    reflect_mission="I am a technical assistant tracking a software team's work."
)

# Seed with diverse data to demonstrate different retrieval strategies
seed_data = [
    # People (keyword search excels here)
    {"content": "Alice Chen is the backend lead, expert in distributed systems", "context": "people"},
    {"content": "Bob Martinez is the frontend lead, specializes in React and TypeScript", "context": "people"},
    {"content": "Carol Wu is the DevOps engineer, manages the Kubernetes clusters", "context": "people"},
    {"content": "Dave Kim is the data engineer, builds ETL pipelines with Apache Spark", "context": "people"},
    
    # Tech stack (semantic search excels: "infrastructure" matches "Kubernetes")
    {"content": "Backend services run on Kubernetes (EKS) with Docker containers", "context": "infrastructure"},
    {"content": "PostgreSQL is the primary database, with Redis for caching", "context": "infrastructure"},
    {"content": "CI/CD pipeline uses GitHub Actions with ArgoCD for deployments", "context": "infrastructure"},
    {"content": "Monitoring stack: Prometheus + Grafana + Loki for logs", "context": "infrastructure"},
    
    # Timeline events (temporal search)
    {"content": "January 2026: Project kickoff and architecture design", "context": "timeline", "timestamp": "2026-01-15T00:00:00Z"},
    {"content": "March 2026: Backend MVP completed with core APIs", "context": "timeline", "timestamp": "2026-03-01T00:00:00Z"},
    {"content": "April 2026: Frontend beta released to 50 test users", "context": "timeline", "timestamp": "2026-04-15T00:00:00Z"},
    {"content": "June 2026: Full production launch, 10k users on day one", "context": "timeline", "timestamp": "2026-06-01T00:00:00Z"},
    
    # Decisions (graph search: entity relationships)
    {"content": "Alice decided to use gRPC for inter-service communication", "context": "decision"},
    {"content": "Bob advocated for Next.js over plain React for SSR benefits", "context": "decision"},
    {"content": "Carol chose EKS over ECS for better multi-region support", "context": "decision"},
    {"content": "Alice and Carol collaborated on the service mesh design with Istio", "context": "decision"},
]

for item in seed_data:
    ts = item.pop("timestamp", None)
    client.retain(bank_id=BANK_ID, timestamp=ts, **item)

print(f"✓ Seeded {len(seed_data)} diverse memories into {BANK_ID}")

## 1. Basic Recall

In [ ]:
# Simple recall — ask naturally, Hindsight figures out the right strategy
results = client.recall(
    bank_id=BANK_ID,
    query="Who handles the frontend?"
)

print(f"Query: 'Who handles the frontend?'")
print(f"Found {len(results.results)} results:\n")
for r in results.results:
    print(f"  [{r.type}] score={r.score:.3f} | {r.text}")

## 2. Budget Levels — Controlling Search Depth

`budget` controls how extensively Hindsight searches:

In [ ]:
# Compare low vs mid vs high budget
for budget in ["low", "mid", "high"]:
    results = client.recall(
        bank_id=BANK_ID,
        query="infrastructure decisions",
        budget=budget
    )
    print(f"\nBudget: {budget} → {len(results.results)} results")
    for r in results.results[:3]:
        print(f"  • {r.text[:80]}...")

## 3. Type Filtering — Focus on Specific Memory Types

Filter by `world`, `experience`, or `observation`.

In [ ]:
# Show what types are available
all_results = client.recall(bank_id=BANK_ID, query="infrastructure", budget="high")
types_found = set(r.type for r in all_results.results)
print(f"Memory types in bank: {types_found}")

In [ ]:
# Filter to only world facts
world_results = client.recall(
    bank_id=BANK_ID,
    query="infrastructure",
    types=["world"],
    budget="high"
)
print(f"World facts only: {len(world_results.results)} results")
for r in world_results.results:
    print(f"  • {r.text[:80]}...")

## 4. Tag-Based Filtering

Tags can be attached during retain or via the API. Filter with `tags` and `tags_match`.

In [ ]:
# Let's add tagged memories
tagged_items = [
    ("Critical security vulnerability found in auth service", ["priority:critical", "area:security"]),
    ("Performance regression in search API — p99 up 300ms", ["priority:high", "area:performance"]),
    ("UI button color mismatch in dark mode", ["priority:low", "area:frontend"]),
    ("Memory leak in data processing pipeline", ["priority:critical", "area:performance"]),
]

for content, tags in tagged_items:
    # Tags are set via metadata or API — here we use retain
    # Note: explicit tag support may vary by API version; metadata works universally
    client.retain(
        bank_id=BANK_ID,
        content=content,
        metadata={t.split(":")[0]: t.split(":")[1] for t in tags}
    )
    
print("✓ Added tagged priority items")

# Now search with metadata filtering for critical items
critical = client.recall(
    bank_id=BANK_ID,
    query="issues",
    metadata_filter={"priority": "critical"}
)
print(f"\nCritical issues: {len(critical.results)} found")
for r in critical.results:
    print(f"  • {r.text}")

## 5. Chunk Retrieval — See the Original Source

`include_chunks=True` returns the original source text that generated each fact.

In [ ]:
# Retrieve with chunks — see the raw source text
response = client.recall(
    bank_id=BANK_ID,
    query="infrastructure choices",
    include_chunks=True,
    max_chunk_tokens=500,
    budget="mid"
)

for r in response.results:
    print(f"Memory: {r.text}")
    if hasattr(r, 'chunks') and r.chunks:
        for chunk in r.chunks[:1]:  # Show first chunk only
            print(f"  Source: {chunk.text[:150]}...")
    print()

## 6. Entity Extraction in Results

`include_entities=True` shows the entities Hindsight extracted from each memory.

In [ ]:
# Get entities along with memories
results = client.recall(
    bank_id=BANK_ID,
    query="Alice",
    include_entities=True,
    budget="high"
)

for r in results.results:
    print(f"Memory: {r.text}")
    if hasattr(r, 'entities') and r.entities:
        entity_names = [e.get('name', str(e)) for e in r.entities]
        print(f"  Entities: {', '.join(entity_names)}")
    print()

## 7. Debugging with Trace Mode

`trace=True` shows what each retrieval strategy contributed.

In [ ]:
# Trace mode — see behind the scenes
results = client.recall(
    bank_id=BANK_ID,
    query="What infrastructure decisions were made?",
    trace=True,
    budget="high"
)

print("=== Results ===")
for r in results.results:
    print(f"  [{r.type}] score={r.score:.3f} | {r.text[:80]}...")

print(f"\n=== Trace ===")
if results.trace:
    for key, value in results.trace.items():
        if isinstance(value, list):
            print(f"  {key}: {len(value)} hits")
        else:
            print(f"  {key}: {value}")
else:
    print("  (no trace data — may not be supported in this API version)")

## 8. Practical Pattern: Context Injection for Agents

The most common pattern: recall relevant memories and inject them into the agent's prompt.

In [ ]:
class MemoryContextProvider:
    """Retrieves relevant memories and formats them as context for an LLM prompt."""
    
    def __init__(self, client, bank_id):
        self.client = client
        self.bank_id = bank_id
    
    def get_context(self, user_query, max_tokens=2048):
        """Retrieve relevant memories and format as context."""
        results = self.client.recall(
            bank_id=self.bank_id,
            query=user_query,
            budget="mid",
            max_tokens=max_tokens
        )
        
        if not results.results:
            return "(No relevant past context found.)"
        
        # Format as bullet points
        context_lines = ["Relevant context from past interactions:"]
        for r in results.results:
            context_lines.append(f"- [{r.type}] {r.text}")
        
        return "\n".join(context_lines)
    
    def build_prompt(self, user_query):
        """Build a complete prompt with injected context."""
        context = self.get_context(user_query)
        return f"""{context}

User query: {user_query}

Using the context above, respond to the user."""

# Usage
provider = MemoryContextProvider(client, BANK_ID)

query = "What's the best approach for deploying the new service?"
prompt = provider.build_prompt(query)
print("=== Generated Prompt ===")
print(prompt)

## 9. Temporal Recall Pattern

In [ ]:
# Temporal queries — Hindsight understands relative time
temporal_queries = [
    "What happened in early 2026?",
    "What was the project status in March?",
    "What happened most recently?",
]

for query in temporal_queries:
    results = client.recall(bank_id=BANK_ID, query=query, budget="mid")
    print(f"\nQuery: '{query}'")
    for r in results.results[:3]:
        print(f"  • {r.text[:80]}...")

## 10. Cleanup

In [ ]:
# client.delete_bank(bank_id=BANK_ID)
print("Bank preserved.")

## Summary

You learned:
1. **Basic recall** — natural language queries
2. **Budget levels** — controlling search depth (low/mid/high)
3. **Type filtering** — world vs experience vs observation
4. **Tag/metadata filtering** — per-user isolation, priority filtering
5. **Chunk retrieval** — seeing original source text
6. **Entity extraction** — getting extracted entities with results
7. **Trace mode** — debugging retrieval behavior
8. **Context injection pattern** — injecting memories into LLM prompts

**Next:** [05_reflect.ipynb](05_reflect.ipynb) — master disposition-aware reasoning